# Walmart MongoDB → PostgreSQL Migration — EDA Notebook

**Author:** Nitin

**Purpose:**
This notebook performs exploratory data analysis (EDA) on the `walmart` MongoDB database
prior to migrating the data into PostgreSQL. The main goals are to:

1. Start and connect to a local MongoDB server instance.
2. Connect to the `walmart` database and verify the connection.
3. List and inspect the available collections in the database.
4. Get a document-count summary for each collection to understand data volume
   and validate completeness before writing the data to PostgreSQL.

**Notes:**
- MongoDB server: local instance (`mongod`) with data directory at `C:\mongodb\data`.
- Database name: `walmart`.
- Downstream step (not covered in this notebook): transform and load these collections
  into corresponding PostgreSQL tables.


In [5]:
# ------------------------------------------------------------------
# Step 1: Start the local MongoDB server (mongod) and confirm it's up
# ------------------------------------------------------------------
import os
import subprocess
import time
from pathlib import Path

from pymongo import MongoClient

# Database directory where MongoDB will store its data files
db_path = r"C:\mongodb\data"
os.makedirs(db_path, exist_ok=True)

# Automatically locate mongod.exe under the MongoDB install directory
server_dir = Path(r"C:\Program Files\MongoDB\Server")
mongod_files = list(server_dir.rglob("mongod.exe"))

if not mongod_files:
    raise FileNotFoundError("mongod.exe not found. Is MongoDB Server installed?")

mongod = str(mongod_files[0])
print(f"Using: {mongod}")

# Launch the MongoDB server as a background process
process = subprocess.Popen(
    [mongod, "--dbpath", db_path],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Client used only to ping the server and confirm it has started
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=1000)

# Retry the ping a few times, since mongod can take a moment to start
for _ in range(10):
    try:
        client.admin.command("ping")
        print("MongoDB started successfully.")
        print(f"PID: {process.pid}")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("MongoDB failed to start.")

Using: C:\Program Files\MongoDB\Server\8.3\bin\mongod.exe
MongoDB started successfully.
PID: 15504


In [2]:
# ------------------------------------------------------------------
# Step 2: Connect to the 'walmart' database
# ------------------------------------------------------------------
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

# Access the database (created lazily by MongoDB if it doesn't already exist)
db = client["walmart"]

print(db.name)

walmart


In [3]:
# ------------------------------------------------------------------
# Step 3: List all collections in the 'walmart' database
# ------------------------------------------------------------------
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["walmart"]

# Get all collection names present in the database
collections = db.list_collection_names()

print(f"Total Collections: {len(collections)}")
print("-" * 40)

for i, name in enumerate(collections, start=1):
    print(f"{i}. {name}")

Total Collections: 6
----------------------------------------
1. employees
2. stores
3. products
4. customers
5. orders
6. order_items


In [4]:
# ------------------------------------------------------------------
# Step 4: Summarize document counts per collection
# This helps validate row/document volumes before the PostgreSQL load
# ------------------------------------------------------------------
import pandas as pd
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["walmart"]

summary = []

# Count documents in each collection and collect into a summary list
for collection in db.list_collection_names():
    summary.append(
        {"Collection": collection, "Documents": db[collection].count_documents({})}
    )

# Display as a DataFrame for a quick, readable overview
pd.DataFrame(summary)

,Collection,Documents
0,employees,250
1,stores,25
2,products,500
3,customers,2000
4,orders,10000
5,order_items,30021
